<a href="https://colab.research.google.com/github/MiguelAtencio/deep-learning-copilot/blob/main/ML4BI_E5_tune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Assignment: Hyperparameter Tuning with Keras Tuner

In this assignment, you'll practice automated hyperparameter tuning using Keras Tuner by building and optimizing a simple neural network model.

### Objective
Create a neural network model using Keras Tuner to systematically search for optimal hyperparameters. Specifically, you will search over:

- **Number of neurons:** Search among reasonable numbers of neurons in the hidden layer.
- **Learning rate:** Search among learning rates of `1e-2`, `1e-3`, or `1e-4`.
- **Architecture:** Use exactly **one hidden layer**.
- **Optimizer:** Use the Adam optimizer.

### Instructions

1. **Data Preparation**: Use Fashion MNIST.

2. **Define the Model**: Use the following template for your `build_model` function:

```python
from tensorflow import keras
from tensorflow.keras import layers

def build_model(hp):
    units = hp.Choice(name="units", values=["YOUR CODE HERE"])
    learning_rate = hp.Choice(name="learning_rate", values=[1e-2, 1e-3, 1e-4])

    model = keras.Sequential([
        layers.Dense(units, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"])

    return model
```

3. **Hyperparameter Search**: Use Keras Tuner (`BayesianOptimization`) to run the search for optimal hyperparameters. Set `keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)` and examine what this does. Record the top 4 configurations.

4. **Optimal number of epochs**: Find the optimal number of epochs using the best training configuration.

5. **Train on the full dataset and test**

## Let's begin

In [3]:
!pip install keras-tuner -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 2.0 MB/s eta 0:00:00a 0:00:01


**Explanation for the number of neurons**: Choosing powers of 2 that are greater than output dimension and limiting it at 256

In [1]:
from tensorflow import keras
from tensorflow.keras import layers

def build_model(hp):
    units = hp.Choice("units", [16, 36, 64, 128, 256])
    learning_rate = hp.Choice("learning_rate", [1e-2, 1e-3, 1e-4])

    model = keras.Sequential([
        layers.Dense(units, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [5]:
import keras_tuner as kt

tuner = kt.BayesianOptimization(
    build_model,
    objective="val_accuracy",
    max_trials=5,
    executions_per_trial=2,
    directory="mnist_kt_test",
    overwrite=True,
)

In [7]:
# 1. Load the Fashion MNIST dataset
import tensorflow as tf
fashion_mnist = tf.keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

# 2. Preprocess the data:
train_images = train_images.reshape(-1, 28*28).astype("float32") / 255.0
test_images = test_images.reshape(-1, 28*28).astype("float32") / 255.0

`keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)` does what?

It multiplies by the factor (0.5) the learning rate if the validation loss hasn't improved in 3 epochs

In [10]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)
]

# Perform search
tuner.search(
    train_images, train_labels,
    batch_size=128,
    epochs=100,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=2
)

Trial 5 Complete [00h 04m 33s]
val_accuracy: 0.8682083189487457

Best val_accuracy So Far: 0.8957083225250244
Total elapsed time: 00h 12m 00s


The pipeline is the following:
1. Search across the given space
2. Get the best models
3. Get the best epoch for the best models
4. Choose a model

In [19]:
top_n = 4
best_hps = tuner.get_best_hyperparameters(top_n)

In [20]:
def get_best_epoch(hp):
    model = build_model(hp)

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=5),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)
    ]
    
    history = model.fit(
      train_images, train_labels,
      batch_size=128,
      epochs=100,
      validation_split=0.2,
      callbacks=callbacks,
      verbose=2
    )
    val_loss_per_epoch = history.history["val_loss"]
    best_epoch = val_loss_per_epoch.index(min(val_loss_per_epoch)) + 1
    print(f"Best epoch: {best_epoch}")
    return best_epoch

In [21]:
def get_best_trained_model(hp):
    best_epoch = get_best_epoch(hp)
    model = build_model(hp)
    model.fit(
        train_images, train_labels,
        batch_size=128, epochs=best_epoch)
    return model

best_models = []
for hp in best_hps:
    model = get_best_trained_model(hp)
    model.evaluate(test_images, test_labels)
    best_models.append(model)

Epoch 1/100
375/375 - 4s - 11ms/step - accuracy: 0.8017 - loss: 0.5787 - val_accuracy: 0.8301 - val_loss: 0.4809 - learning_rate: 0.0010
Epoch 2/100
375/375 - 2s - 5ms/step - accuracy: 0.8534 - loss: 0.4163 - val_accuracy: 0.8610 - val_loss: 0.3909 - learning_rate: 0.0010
Epoch 3/100
375/375 - 2s - 5ms/step - accuracy: 0.8664 - loss: 0.3744 - val_accuracy: 0.8593 - val_loss: 0.3877 - learning_rate: 0.0010
Epoch 4/100
375/375 - 2s - 5ms/step - accuracy: 0.8742 - loss: 0.3490 - val_accuracy: 0.8695 - val_loss: 0.3731 - learning_rate: 0.0010
Epoch 5/100
375/375 - 2s - 5ms/step - accuracy: 0.8818 - loss: 0.3302 - val_accuracy: 0.8732 - val_loss: 0.3539 - learning_rate: 0.0010
Epoch 6/100
375/375 - 2s - 5ms/step - accuracy: 0.8873 - loss: 0.3115 - val_accuracy: 0.8816 - val_loss: 0.3374 - learning_rate: 0.0010
Epoch 7/100
375/375 - 3s - 7ms/step - accuracy: 0.8908 - loss: 0.2978 - val_accuracy: 0.8824 - val_loss: 0.3350 - learning_rate: 0.0010
Epoch 8/100
375/375 - 2s - 5ms/step - accuracy:

In [22]:
best_models

[<Sequential name=sequential_2, built=True>,
 <Sequential name=sequential_4, built=True>,
 <Sequential name=sequential_6, built=True>,
 <Sequential name=sequential_8, built=True>]

In [23]:
for idx, model in enumerate(best_models):
    loss, acc = model.evaluate(test_images, test_labels, verbose=0)
    print(f"Model {idx+1}: Test Loss = {loss:.4f}, Test Accuracy = {acc:.4f}")


Model 1: Test Loss = 0.3553, Test Accuracy = 0.8838
Model 2: Test Loss = 0.4063, Test Accuracy = 0.8703
Model 3: Test Loss = 0.4223, Test Accuracy = 0.8535
Model 4: Test Loss = 0.3952, Test Accuracy = 0.8630


In [24]:
import pandas as pd

results = []
for idx, (hp, model) in enumerate(zip(best_hps, best_models)):
    loss, acc = model.evaluate(test_images, test_labels, verbose=0)
    results.append({
        "Model": idx+1,
        "Units": hp.get("units"),
        "Learning Rate": hp.get("learning_rate"),
        "Test Accuracy": f"{acc*100:.2f}%",
        "Test Loss": f"{loss:.3f}"
    })

df_results = pd.DataFrame(results)
print(df_results)

   Model  Units  Learning Rate Test Accuracy Test Loss
0      1    128          0.001        88.38%     0.355
1      2    128          0.010        87.03%     0.406
2      3     64          0.010        85.35%     0.422
3      4     16          0.001        86.30%     0.395
